In [4]:
from ultralytics import YOLO




In [16]:
model = YOLO("yolov8n.pt")


In [2]:
import os
import xml.etree.ElementTree as ET
import cv2


# Input and output folders
annotations_dir = "annotations"
images_dir = "images"
labels_dir = "labels"

# Create output dir if not exist
os.makedirs(labels_dir, exist_ok=True)

# Gather all unique class names
classes = set()

# First pass: get all classes
for xml_file in os.listdir(annotations_dir):
    if not xml_file.endswith('.xml'):
        continue
    tree = ET.parse(os.path.join(annotations_dir, xml_file))
    root = tree.getroot()
    for obj in root.findall('object'):
        cls = obj.find('name').text
        classes.add(cls)

# Save classes in fixed order
classes = sorted(classes)
with open("classes.txt", "w") as f:
    for cls in classes:
        f.write(cls + "\n")

# Map class name to index
class_to_idx = {cls: i for i, cls in enumerate(classes)}

# Convert each annotation
for xml_file in os.listdir(annotations_dir):
    if not xml_file.endswith('.xml'):
        continue

    tree = ET.parse(os.path.join(annotations_dir, xml_file))  
    root = tree.getroot()

    size = root.find("size")
    w = int(size.find("width").text)
    h = int(size.find("height").text)

    yolo_lines = []

    for obj in root.findall("object"):
        cls = obj.find("name").text
        class_id = class_to_idx[cls]

        bndbox = obj.find("bndbox")
        xmin = int(bndbox.find("xmin").text)
        ymin = int(bndbox.find("ymin").text)
        xmax = int(bndbox.find("xmax").text)
        ymax = int(bndbox.find("ymax").text)

        # Convert to YOLO format
        x_center = (xmin + xmax) / 2.0 / w
        y_center = (ymin + ymax) / 2.0 / h
        box_width = (xmax - xmin) / w
        box_height = (ymax - ymin) / h

        yolo_lines.append(f"{class_id} {x_center:.6f} {y_center:.6f} {box_width:.6f} {box_height:.6f}")

    # Write label file
    txt_filename = os.path.splitext(xml_file)[0] + ".txt"
    with open(os.path.join(labels_dir, txt_filename), "w") as f:
        f.write("\n".join(yolo_lines))


In [1]:
pip show ultralytics



SyntaxError: invalid syntax (2686303703.py, line 1)

In [2]:
import sys
print(sys.executable)


/home/sandeep/anaconda3/bin/python


In [4]:
import os
import random
import shutil

# Paths
image_dir = 'images'        # or './images'
label_dir = 'labels'        # or './labels'

train_img_dir = 'images/train'
val_img_dir = 'images/val'
train_lbl_dir = 'labels/train'
val_lbl_dir = 'labels/val'


# Create output directories..
os.makedirs(train_img_dir, exist_ok=True)
os.makedirs(val_img_dir, exist_ok=True)
os.makedirs(train_lbl_dir, exist_ok=True)
os.makedirs(val_lbl_dir, exist_ok=True)

# Get all image files..
image_files = [f for f in os.listdir(image_dir) if f.endswith(('.jpg', '.png'))]

# Shuffle and split
random.shuffle(image_files)
split_idx = int(0.8 * len(image_files))
train_files = image_files[:split_idx]
val_files = image_files[split_idx:]

# Function to move both image and label
def move_files(files, img_dest, lbl_dest):
    for file in files:
        img_src = os.path.join(image_dir, file)
        lbl_src = os.path.join(label_dir, os.path.splitext(file)[0] + '.txt')
        shutil.copy2(img_src, os.path.join(img_dest, file))
        shutil.copy2(lbl_src, os.path.join(lbl_dest, os.path.splitext(file)[0] + '.txt'))

# Movefiles
move_files(train_files, train_img_dir, train_lbl_dir)
move_files(val_files, val_img_dir, val_lbl_dir)

print(f"Split complete. {len(train_files)} training and {len(val_files)} validation samples.")


Split complete. 682 training and 171 validation samples.


In [17]:
model.model.fuse()
for i, p in enumerate(model.model.parameters()):
    if i < len(list(model.model.parameters())) - 2:
        p.requires_grad = False




model.train(data="dataset.yaml", epochs=15, imgsz=416,batch=8)

YOLOv8n summary (fused): 72 layers, 3,151,904 parameters, 0 gradients
Ultralytics 8.3.169 🚀 Python-3.11.7 torch-2.7.1+cpu CPU (unknown)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=dataset.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=15, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=416, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train6, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.

train: Scanning /home/sandeep/Downloads/tfprof/labels/train.cache... 682 images,

val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1755.0±77.4 MB/s, size: 547.1 KB)



/home/sandeep/anaconda3/lib/python3.11/site-packages/torch/utils/data/dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
val: Scanning /home/sandeep/Downloads/tfprof/labels/val.cache... 171 images, 0 b

Plotting labels to runs/detect/train6/labels.jpg... 


...

                   all        171        843      0.841      0.576      0.668      0.417

15 epochs completed in 0.305 hours.


Optimizer stripped from runs/detect/train6/weights/last.pt, 6.2MB
Optimizer stripped from runs/detect/train6/weights/best.pt, 6.2MB

Validating runs/detect/train6/weights/best.pt...
Ultralytics 8.3.169 🚀 Python-3.11.7 torch-2.7.1+cpu CPU (unknown)
Model summary (fused): 72 layers, 3,006,233 parameters, 0 gradients


                 Class     Images  Instances      Box(P          R      mAP50  m


                   all        171        843      0.841      0.576      0.668      0.417
 mask_weared_incorrect         22         27      0.872      0.296      0.432      0.286
             with_mask        153        686      0.904      0.816      0.897      0.578
          without_mask         56        130      0.748      0.617      0.674      0.386
Speed: 0.2ms preprocess, 17.6ms inference, 0.0ms loss, 0.4ms postprocess per image
Results saved to runs/detect/train6


ModuleNotFoundError: No module named 'scipy'

In [20]:
import os
import random
import matplotlib.pyplot as plt
val_img_dir = "images/val"  
random_img_path = os.path.join(val_img_dir, random.choice(os.listdir(val_img_dir)))


results = model(random_img_path)

#Showresult..
results[0].show()  


image 1/1 /home/sandeep/Downloads/tfprof/images/val/maksssksksss439.png: 448x640 4 with_masks, 47.2ms
Speed: 1.1ms preprocess, 47.2ms inference, 0.5ms postprocess per image at shape (1, 3, 448, 640)
